### This notebook aims to allow easy integration of new domains in the framework 

In [1]:
import argparse
import numpy as np
import random as rd 
import os
import soundfile as sf
import torch
from pathlib import Path
import json 
import torchaudio
import scipy as sp

/home/fderrida/miniconda3/envs/domainbed_old/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Inpainting corruption

In [2]:
def load_audio(
    path,
    transforms=None,   # e.g. [Resample(16000), Normalize()]
):
    """
    Load audio with soundfile and apply torch-based transforms.
    Returns: (waveform, sr) where waveform is 1D torch.Tensor
    """
    
    # Load with soundfile
    audio, sr = sf.read(path, dtype="float32")
    # assert sr == sample_rate, f"Expected {sample_rate}, got {sr}"

    # Convert to mono if needed
    if audio.ndim > 1:
        audio = np.mean(audio, axis=1)

    # To torch tensor (shape: [T])
    waveform = torch.from_numpy(audio)

    data = (waveform, sr)

    # Apply transforms sequentially
    if transforms is not None:
        for t in transforms:
            data = t(data)

    return data

class Resample:
    def __init__(self, target_sr):
        self.target_sr = target_sr

    def __call__(self, data):
        waveform, sr = data

        if sr != self.target_sr:
            waveform = torchaudio.functional.resample(
                waveform,
                orig_freq=sr,
                new_freq=self.target_sr
            )

        return waveform, self.target_sr

class Normalize:
    def __call__(self, data):
        waveform, sr = data
        max_val = waveform.abs().max().clamp(min=1e-6)
        waveform = waveform / max_val

        return waveform, sr
    
SAMPLE_RATE = 16000

def add_saturation(x, gain):
    """
    Works for [T] or [B,1,T]
    """

    if x.dim() == 1:
        rms = torch.sqrt(torch.mean(x**2))
        x_norm = x / (rms + 1e-8)
    else:
        rms = torch.sqrt(torch.mean(x**2, dim=-1, keepdim=True))
        x_norm = x / (rms + 1e-8)

    x_amp = gain * x_norm
    # sat_x = torch.clamp(x_amp, -1.0, 1.0)
    sat_x = torch.tanh(x_amp)

    return sat_x



def add_inpainting(x, ratio, n_gaps=10):
    ratio = max(0.0, min(1.0, float(ratio)))
    y = x.clone()

    def mask_signal(sig):
        T = sig.shape[-1]
        total_missing = int(ratio * T)

        if total_missing == 0:
            return sig

        # Divide the missing samples across the gaps
        lengths = torch.full((n_gaps,), total_missing // n_gaps, dtype=torch.long)
        lengths[:total_missing % n_gaps] += 1

        for L in lengths:
            L = int(L)
            if L == 0:
                continue

            start = torch.randint(0, T - L + 1, (1,)).item()
            sig[start:start + L] = 0

        return sig

    if x.ndim == 1:
        y = mask_signal(y)
    else:
        for b in range(x.shape[0]):
            y[b, 0] = mask_signal(y[b, 0])

    return y

In [3]:
transforms = [
        Resample(SAMPLE_RATE),
        Normalize()
    ]
    
clean_path = './eng_S0_2yckAilrWqU_0043-230_0057-300.flac'

clean, sr = load_audio(clean_path, transforms)


mixed = add_saturation(clean, 10)
out_path = './sat_100.flac'
sf.write(
        out_path,
        mixed.cpu().numpy(),
        sr,
        format="flac"
        )


In [4]:
mixed = add_inpainting(clean, 0.2)

out_path = './inpainting.flac'
sf.write(
        out_path,
        mixed.cpu().numpy(),
        sr,
        format="flac"
        )

## LPC10 codec corruption

In [5]:
# sudo apt install sox
!sox --version

sox:      SoX v14.4.2


In [6]:
import sox
import subprocess
import tempfile
import os

clean_path = './eng_S0_2yckAilrWqU_0043-230_0057-300.flac'
clean_name = clean_path.split('.')[1].split('/')[1]



def flac_to_lpc10_to_wav(input_file, output_file):

    clean_name = input_file.split('.')[1].split('/')[1]

    with tempfile.NamedTemporaryFile(suffix=".lpc", delete=False) as tmp:
        lpc_file = tmp.name

    try:
        # FLAC → LPC-10
        subprocess.run([
            "sox",
            input_file,
            "-t", "lpc10",
            lpc_file
        ], check=True)

        # LPC-10 → WAV
        subprocess.run([
            "sox",
            "-t", "lpc10",
            lpc_file,
            f"{clean_name}_lpc10.wav"
        ], check=True)

    finally:
        # Always remove the intermediate LPC-10 file
        if os.path.exists(lpc_file):
            os.remove(lpc_file)



flac_to_lpc10_to_wav(
clean_path,
"output.wav"
)


sox WARN formats: lpc10 can't encode at 16000Hz; using 8000Hz
